In [ ]:
#also LSTM based model training with sliding window
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5

In [3]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

In [5]:
messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')
messages_with_lables_df['text'] = messages_with_lables_df.apply(lambda x: f"פונה: {x['text']}" if x['seeker'] else f"נציג: {x['text']}",axis=1)
messages_with_lables_df['anonymized'] = messages_with_lables_df.apply(lambda x: f"פונה: {x['anonymized']}" if x['seeker'] else f"נציג: {x['anonymized']}",axis=1)


#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

In [6]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df['text'] = all_messages_df.apply(lambda x: f"פונה: {x['text']}" if x['seeker'] else f"נציג: {x['text']}",axis=1)
all_messages_df['anonymized'] = all_messages_df.apply(lambda x: f"פונה: {x['anonymized']}" if x['seeker'] else f"נציג: {x['anonymized']}",axis=1)

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


In [7]:
# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_weighted_train_loaders(dfs, weights, tokenize):
    #make sampled df
    acc_train = pd.DataFrame(data= {})
    for df, weight in zip(dfs,weights):
        sample = df.sample(frac = weight)
        acc_train = pd.concat([acc_train,sample[["anonymized" , "label"]]])
    
    #create datasets
    train_dataset = Dataset.from_pandas(acc_train)
    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader

def make_test_loader(df, tokenize):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return test_loader

In [8]:
#labled_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [1,0], tokenize)
#all_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [0,1], tokenize)
labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)
all_messages_test_loader = make_test_loader(test_all_messages_df, tokenize)

Map: 100%|██████████| 6047/6047 [00:08<00:00, 716.46 examples/s]


In [9]:
def general_trainer(train_loader, loss_fn=None):
    optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
    bert_model.train()
    
    total_batches = len(train_loader)
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        if loss_fn is None:
            loss = outputs.loss
        else:
            loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")

In [10]:
plan = [
    [0.9,0.1],
    [0.8,0.2],
    [0.65,0.3],
    [0.1,0.7],
    [0.05,0.9],
    [0.025,0.95]
]
for w in plan:
    train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , w, tokenize)
    general_trainer(train_loader)

Map: 100%|██████████| 295327/295327 [01:34<00:00, 3124.49 examples/s]


Batch 1000/18458 completed. 17458 remaining.
Batch 2000/18458 completed. 16458 remaining.
Batch 3000/18458 completed. 15458 remaining.
Batch 4000/18458 completed. 14458 remaining.
Batch 5000/18458 completed. 13458 remaining.
Batch 6000/18458 completed. 12458 remaining.
Batch 7000/18458 completed. 11458 remaining.
Batch 8000/18458 completed. 10458 remaining.
Batch 9000/18458 completed. 9458 remaining.
Batch 10000/18458 completed. 8458 remaining.
Batch 11000/18458 completed. 7458 remaining.
Batch 12000/18458 completed. 6458 remaining.
Batch 13000/18458 completed. 5458 remaining.
Batch 14000/18458 completed. 4458 remaining.
Batch 15000/18458 completed. 3458 remaining.
Batch 16000/18458 completed. 2458 remaining.
Batch 17000/18458 completed. 1458 remaining.
Batch 18000/18458 completed. 458 remaining.


Map: 100%|██████████| 265200/265200 [01:26<00:00, 3055.32 examples/s]


Batch 1000/16575 completed. 15575 remaining.
Batch 2000/16575 completed. 14575 remaining.
Batch 3000/16575 completed. 13575 remaining.
Batch 4000/16575 completed. 12575 remaining.
Batch 5000/16575 completed. 11575 remaining.
Batch 6000/16575 completed. 10575 remaining.
Batch 7000/16575 completed. 9575 remaining.
Batch 8000/16575 completed. 8575 remaining.
Batch 9000/16575 completed. 7575 remaining.
Batch 10000/16575 completed. 6575 remaining.
Batch 11000/16575 completed. 5575 remaining.
Batch 12000/16575 completed. 4575 remaining.
Batch 13000/16575 completed. 3575 remaining.
Batch 14000/16575 completed. 2575 remaining.
Batch 15000/16575 completed. 1575 remaining.
Batch 16000/16575 completed. 575 remaining.


Map: 100%|██████████| 218801/218801 [01:15<00:00, 2910.91 examples/s]


...

Batch 1000/3093 completed. 2093 remaining.
Batch 2000/3093 completed. 1093 remaining.
Batch 3000/3093 completed. 93 remaining.


Map: 100%|██████████| 38039/38039 [00:34<00:00, 1094.57 examples/s]


Batch 1000/2378 completed. 1378 remaining.
Batch 2000/2378 completed. 378 remaining.


Map: 100%|██████████| 31112/31112 [00:33<00:00, 916.99 examples/s]


Batch 1000/1945 completed. 945 remaining.


In [15]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
# Model
def sliding_window_inference(
    conversation,
    tokenizer,
    model,
    max_length=512,
    stride=256,
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    """
    Performs inference on a conversation using sliding window if needed.

    Args:
        conversation (str): The full conversation text.
        tokenizer: HuggingFace tokenizer.
        model: Trained HuggingFace classification model.
        max_length (int): Max tokens per chunk.
        stride (int): Stride for overlapping chunks.
        device (str): 'cuda' or 'cpu'.

    Returns:
        List[Tensor]: Softmax probabilities per chunk (or one if short).
    """
    model.eval()
    model.to(device)

    # Tokenize without truncation to get full length
    tokens = tokenizer(conversation, return_tensors='pt', truncation=False, add_special_tokens=False)
    input_ids = tokens['input_ids'][0]
    total_len = input_ids.shape[0]

    predictions = []

    # Case 1: The whole conversation fits in one chunk
    if total_len <= max_length:
        padded_input_ids = torch.cat([
            input_ids,
            torch.full((max_length - total_len,), tokenizer.pad_token_id)
        ])
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()

        with torch.no_grad():
            outputs = model(
                input_ids=padded_input_ids.unsqueeze(0).to(device),
                attention_mask=attention_mask.unsqueeze(0).to(device)
            )
            logits = outputs.logits.squeeze(0).cpu()
            predictions.append(torch.softmax(logits, dim=-1))

    # Case 2: Conversation is longer than max_length → sliding window
    else:
        for start_idx in range(0, total_len, stride):
            end_idx = start_idx + max_length
            chunk_ids = input_ids[start_idx:end_idx]

            # Pad if shorter than max_length
            if chunk_ids.shape[0] < max_length:
                padding_len = max_length - chunk_ids.shape[0]
                chunk_ids = torch.cat([
                    chunk_ids,
                    torch.full((padding_len,), tokenizer.pad_token_id)
                ])

            attention_mask = (chunk_ids != tokenizer.pad_token_id).long()

            with torch.no_grad():
                outputs = model(
                    input_ids=chunk_ids.unsqueeze(0).to(device),
                    attention_mask=attention_mask.unsqueeze(0).to(device)
                )
                logits = outputs.logits.squeeze(0).cpu()
                predictions.append(torch.softmax(logits, dim=-1))

    return predictions

class RiskAggregator(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )
    def forward(self, padded_seq, lengths):
        packed = pack_padded_sequence(padded_seq, lengths, batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.lstm(packed)
        return self.classifier(hn[-1]).squeeze(1)

# Dataset
class ConversationDataset(Dataset):
    def __init__(self, df, tokenizer, model, device):
        self.data = []
        self.labels = []
        model.eval()
        for _, row in df.iterrows():
            conv = row["anonymized"]
            predictions = sliding_window_inference(conv, tokenizer, model)
            p1s = [float(x[1]) for x in predictions]
            self.data.append(torch.tensor(p1s))
            self.labels.append(float(row["label"]))
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

# Collate function
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True).unsqueeze(-1)
    return padded.to(device), lengths.cpu().long(), torch.tensor(labels).to(device)



In [16]:
# Load data
train_dataset = ConversationDataset(train_all_messages_df, tokenizer, bert_model, device)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

Token indices sequence length is longer than the specified maximum sequence length for this model (1263 > 512). Running this sequence through the model will result in indexing errors


In [17]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RiskAggregator().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# Train
model.train()
for epoch in range(6):
    total_loss = 0
    for padded, lengths, labels in train_loader:
        optimizer.zero_grad()
        preds = model(padded, lengths)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 701.9324
Epoch 2: Loss = 427.5191
Epoch 3: Loss = 403.0760
Epoch 4: Loss = 390.3574
Epoch 5: Loss = 383.9287
Epoch 6: Loss = 380.4941


In [18]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

In [19]:
labels = []
preds = []
pred_probs = []

model.eval()
with torch.no_grad():
    for index, row in test_all_messages_df.iterrows():
        conv = row["anonymized"]
        predictions = sliding_window_inference(
            conversation=conv,
            tokenizer=tokenizer,
            model=bert_model
        )
        sucidal_preds = [float(x[1]) for x in predictions]

        # Convert to tensor and run through the model
        seq_tensor = torch.tensor(sucidal_preds, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)  # (1, seq_len, 1)
        length_tensor = torch.tensor([len(sucidal_preds)]).cpu().long()
        
        if len(seq_tensor[0]) > 1:
            output = model(seq_tensor, length_tensor)  # shape: (1,)
            prob = output.item()
        else:
            prob = seq_tensor[0].item()

        
        labels.append(row["label"])
        preds.append(int(prob > 0.5))
        pred_probs.append(prob)

# Final evaluation
calc_matrics(labels, preds, pred_probs)

Accuracy: 0.8939970233173474
Precision: 0.7316326530612245
Recall: 0.6547945205479452
F1: 0.6910843373493976
ROC-AUC: 0.9182296161875464
F2: 0.6688432835820896


In [20]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.9520469510449471
Precision: 0.7848422579475401
Recall: 0.7435016603687163
F1: 0.7636128425261672
ROC-AUC: 0.9735204707464684
F2: 0.7514176599930563
